# AI Cycling Coach — GPU Training (Kaggle)

**Settings (right sidebar) → Accelerator → GPU T4 x2** before running.

Then click **Run All**. No uploads needed — generates data here (~3 min) then trains on GPU (~1–2 h).

In [ ]:
# ── 1. Check GPU ─────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('No GPU — go to Settings → Accelerator → GPU T4 x2')

In [ ]:
# ── 2. Clone repo + set paths ────────────────────────────────────────────────
import os, sys

REPO    = 'https://github.com/yossibello/ai-coach.git'
WORKDIR = '/kaggle/working/ai-coach'

if not os.path.exists(WORKDIR):
    !git clone {REPO} {WORKDIR}
else:
    !cd {WORKDIR} && git pull

%cd {WORKDIR}

for p in [f'{WORKDIR}/backend', WORKDIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ['PYTHONPATH']      = f'{WORKDIR}/backend'
os.environ['PYTHONIOENCODING'] = 'utf-8'

!mkdir -p ml/data backend/models
print('cwd:', os.getcwd())
print('sys.path[0:3]:', sys.path[:3])

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
!pip install pyarrow --upgrade -q
import torch, pandas
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
print('pandas:', pandas.__version__)

In [ ]:

# ── 4. Load training data ────────────────────────────────────────────────────
# TWO OPTIONS — set MODE below:
#
#   'dataset'  → you added the ai-coach-synthetic Kaggle dataset (recommended, instant)
#   'generate' → generate here in Kaggle (~3 min for 20K, ~8 min for 50K)

import os, sys, subprocess, multiprocessing, pandas as pd

MODE          = 'generate'  # ← 'dataset' | 'generate'
ATHLETES      = 50_000      # only used when MODE='generate'
DATASET_PATH  = '/kaggle/input/ai-coach-synthetic/synthetic.parquet'
DATA_FILE     = 'ml/data/synthetic.parquet'

os.makedirs('ml/data', exist_ok=True)

if MODE == 'dataset':
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(
            f'Dataset not found at {DATASET_PATH}\n'
            'Add it: right panel → Add data → search "ai-coach-synthetic"'
        )
    print(f'Using Kaggle dataset: {DATASET_PATH}  ({os.path.getsize(DATASET_PATH)/1e6:.0f} MB)')
    DATA_FILE = DATASET_PATH

elif MODE == 'generate':
    workers = max(1, multiprocessing.cpu_count() - 1)
    print(f'Generating {ATHLETES:,} athletes using {workers} workers…')

    # Use subprocess so we capture stdout+stderr and get a real exception on failure.
    # The !shell magic swallows errors and leaves DATA_FILE missing.
    env = os.environ.copy()
    env['PYTHONPATH'] = f"{os.getcwd()}/backend:{os.getcwd()}"
    result = subprocess.run(
        [sys.executable, '-m', 'ml.training.generate_synthetic',
         '--athletes', str(ATHLETES),
         '--workers',  str(workers),
         '--output',   DATA_FILE],
        env=env,
        capture_output=False,   # stream output live to the cell
    )
    if result.returncode != 0:
        raise RuntimeError(
            f'generate_synthetic failed (exit {result.returncode}).\n'
            'Scroll up for the traceback printed above.'
        )

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f'Expected {DATA_FILE} but it was not created.\n'
        'Check the output above for errors.'
    )

df = pd.read_parquet(DATA_FILE)
assert 'risk_ot_class'   in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
assert 'risk_inj_target' in df.columns, 'Old parquet — regenerate with latest generate_synthetic.py'
print(f'✓ Data ready: {len(df):,} rows, {df.athlete_id.nunique():,} athletes, {len(df.columns)} cols')
del df


In [ ]:

# ── 5. Train ─────────────────────────────────────────────────────────────────
import sys, os, torch, argparse

n_gpus  = torch.cuda.device_count() if torch.cuda.is_available() else 1
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

# Per-GPU batch size tuned to fill ~10-11 GB of T4's 15 GB.
# At d_model=256, 8 layers, seq_len=90 with AMP:
#   512/GPU  → ~5 GB  (old — GPU was half-idle)
#   1024/GPU → ~10 GB (new — fills GPU properly, 2× throughput)
if   vram_gb >= 45: per_gpu = 4096   # A6000 / A100 80 GB
elif vram_gb >= 38: per_gpu = 2048   # A100 40 GB / H100
elif vram_gb >= 20: per_gpu = 1024   # RTX 3090+
else:               per_gpu = 1024   # T4 16 GB — was 512, now fills ~10/15 GB

BATCH_SIZE = per_gpu * n_gpus   # e.g. 2×T4 → 1024×2 = 2048 total (1024/GPU)

# 50K athletes → ~14M rows → ~11M sequences → ~5400 batches at bs=2048.
# 5000 steps/epoch covers 93% of data per epoch (was 3000 = 56%).
# Validation still runs on the full val split.
STEPS_PER_EPOCH = 5000   # set None to use all batches

EPOCHS     = 50
MODEL_FILE = 'backend/models/cycling_coach.pt'

# More DataLoader workers → GPU stays fed between batches.
# 6 workers on Kaggle's 4-core CPU (2 hyperthreads each = 8 logical).
os.environ['DATALOADER_WORKERS'] = '6'

# LR scales with batch size (linear rule: 2× batch → 2× LR).
# Old: 3e-4 @ bs=1024. New: 6e-4 @ bs=2048.
LEARNING_RATE = 6e-4

print(f'GPUs: {n_gpus}  |  VRAM/GPU: {vram_gb:.1f} GB  |  batch: {BATCH_SIZE} ({per_gpu}/GPU)')
print(f'Steps/epoch: {STEPS_PER_EPOCH}  |  LR: {LEARNING_RATE}  |  Workers: {os.environ["DATALOADER_WORKERS"]}')
print(f'Data: {DATA_FILE}  ({os.path.getsize(DATA_FILE)/1e6:.0f} MB)')
print('─' * 60)

from ml.training.train import train as run_training

os.makedirs(os.path.dirname(MODEL_FILE), exist_ok=True)

args = argparse.Namespace(
    data             = DATA_FILE,
    output           = MODEL_FILE,
    checkpoint       = None,
    epochs           = EPOCHS,
    batch_size       = BATCH_SIZE,
    steps_per_epoch  = STEPS_PER_EPOCH,
    lr               = LEARNING_RATE,
    seq_len          = 90,
    val_frac         = 0.1,
    seed             = 42,
    d_model          = 256,
    nhead            = 8,
    num_layers       = 8,
    d_ff             = 1024,
    dropout          = 0.1,
    fast             = False,
    patience         = 20,
    compile          = False,
    no_amp           = False,
)

run_training(args)
print(f'\n✓ Training complete! Model → {MODEL_FILE}')


In [ ]:
# ── 6. Copy model to /kaggle/working/ so Kaggle saves it as output ────────────
import shutil, os

OUT = '/kaggle/working/cycling_coach.pt'
shutil.copy(MODEL_FILE, OUT)
print(f'✓ Model saved to {OUT}')
print('  → After the notebook finishes, go to the Output tab and download it.')

In [ ]:
# ── 7. (Optional) Push model to GitHub ──────────────────────────────────────
# Create a PAT at https://github.com/settings/tokens (Classic, repo scope)
# then paste it when prompted.

from getpass import getpass
token = getpass('GitHub PAT (hidden): ')

!git config user.email 'kaggle@training'
!git config user.name  'Kaggle Training'
!git remote set-url origin https://{token}@github.com/yossibello/ai-coach.git
!cp /kaggle/working/cycling_coach.pt backend/models/cycling_coach.pt
!git add backend/models/cycling_coach.pt
!git commit -m "Trained model: {ATHLETES} athletes, {EPOCHS} epochs (Kaggle GPU)"
!git push origin main
print('✓ Model pushed to GitHub!')

In [ ]:
# ── 8. Sanity check ──────────────────────────────────────────────────────────
import torch, sys
from app.ml.model import CyclingTransformer

ckpt = torch.load(MODEL_FILE, map_location='cpu')
cfg  = ckpt.get('config', {})
m    = CyclingTransformer(
    d_model=cfg.get('d_model', 128),
    nhead=cfg.get('nhead', 8),
    num_layers=cfg.get('num_layers', 6),
    dim_feedforward=cfg.get('dim_feedforward', 512),
)
m.load_state_dict(ckpt['state_dict'])
m.eval()
print('Model loaded OK')
print('Params:', sum(p.numel() for p in m.parameters()))
print('Best val loss:', ckpt.get('metrics', {}).get('val_loss', 'n/a'))
print('Epoch:',        ckpt.get('metrics', {}).get('epoch',    'n/a'))